<a href="https://colab.research.google.com/github/vaishnavi29pandey-beep/vaishnavi_53/blob/main/Gym_Membership.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""Gym & Fitness Center Membership Analytics - tasks 1-15."""
"""
HOW TO RUN IN VS CODE
1. Put this file and gym_new_dataset.csv in the same folder.
2. Open that folder in VS Code (File > Open Folder).
3. Install the libraries once, in the VS Code terminal:  pip install pandas numpy
4. Press the "Run Python File" (play) button, or run:  python gym_analysis.py
Results print in the terminal; CSV outputs are saved in an "outputs" folder next to this file.
Optional: pass a different CSV path as the first argument.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Paths are relative to this script, so it works no matter where VS Code's terminal is opened.
BASE_DIR = Path.cwd()
path = BASE_DIR / "gym_new_dataset.csv"
if not path.exists():
    sys.exit(f"CSV not found:/content/gym_new_dataset.csv\nPut gym_new_dataset.csv in the same folder as this script.")

out_dir = BASE_DIR / "outputs"
out_dir.mkdir(exist_ok=True)
out = str(out_dir)

# Show full tables in the terminal instead of truncating columns
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

# 1-2. Load, inspect, quality checks
df = pd.read_csv(path)
print("== 1. SHAPE/DTYPES =="); print(df.shape); print(df.dtypes)
print("== 2. MISSING =="); print(df.isna().sum())
print("Duplicate Checkin_ID:", df.Checkin_ID.duplicated().sum(),
      "| fully duplicated rows:", df.duplicated().sum())

# 3. Impute Workout_Duration_Mins with the MEDIAN of the member's tier.
#    Median is robust to outliers; distribution is near-symmetric (skew ~0.18) and
#    only 3.5% missing, so tier-median keeps the mean/spread essentially unchanged.
print("== 3. Median duration by tier (pre-imputation) ==");
print(df.groupby("Membership_Tier").Workout_Duration_Mins.median())
n_missing = df.Workout_Duration_Mins.isna().sum()
df["Workout_Duration_Mins"] = df.Workout_Duration_Mins.fillna(
    df.groupby("Membership_Tier").Workout_Duration_Mins.transform("median"))
print("Imputed:", n_missing, "-> remaining missing:", df.Workout_Duration_Mins.isna().sum())
# Missing Fitness_Class = visit without a booked class
df["Fitness_Class"] = df.Fitness_Class.fillna("No Class (Open Gym)")

# 4. Datetime features
df["Checkin_Time"] = pd.to_datetime(df.Checkin_Time)
df["Weekday"] = df.Checkin_Time.dt.day_name()
df["Hour"] = df.Checkin_Time.dt.hour
df["Month"] = df.Checkin_Time.dt.to_period("M")
print("Date range:", df.Checkin_Time.min(), "->", df.Checkin_Time.max())

# 5. Totals
print("== 5. TOTALS ==")
print("Total check-ins:", len(df), "| unique members:", df.Member_ID.nunique())
print("Avg workout (mins):", round(df.Workout_Duration_Mins.mean(), 2))

# 6. Tier comparison
tier = df.groupby("Membership_Tier").agg(
    checkins=("Checkin_ID", "count"),
    members=("Member_ID", "nunique"),
    avg_duration=("Workout_Duration_Mins", "mean"),
    median_duration=("Workout_Duration_Mins", "median"),
).round(2)
tier["visits_per_member"] = (tier.checkins / tier.members).round(2)
tier["share_of_checkins_%"] = (100 * tier.checkins / tier.checkins.sum()).round(1)
tier = tier.reindex(["Basic", "Gold", "Platinum"])
print("== 6. TIER =="); print(tier)

# 7. Peak hours / days
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_hour = df.groupby("Hour").size()
by_day = df.groupby("Weekday").size().reindex(order)
print("== 7. BY HOUR =="); print(by_hour.to_string())
print("Top 5 hours:", by_hour.sort_values(ascending=False).head(5).to_dict())
print("BY DAY:"); print(by_day.to_string())
band = pd.cut(df.Hour, [-1, 5, 8, 11, 15, 18, 21, 23],
              labels=["<=05", "06-08", "09-11", "12-15", "16-18", "19-21", "22-23"])
print("Hour bands:"); print(df.groupby(band, observed=True).size().to_string())
heat = df.pivot_table(index="Weekday", columns="Hour", values="Checkin_ID", aggfunc="count", fill_value=0).reindex(order)
heat.to_csv(f"{out}/heatmap_day_hour.csv")

# 8. Class ranking
cls = df.groupby("Fitness_Class").agg(attendance=("Checkin_ID", "count"),
                                      avg_duration=("Workout_Duration_Mins", "mean")).round(1)
cls["share_%"] = (100 * cls.attendance / cls.attendance.sum()).round(1)
cls = cls.sort_values("attendance", ascending=False)
print("== 8. CLASSES =="); print(cls)

# 9. Gender
print("== 9. GENDER =="); print(df.groupby("Gender").Workout_Duration_Mins.agg(["count", "mean", "median"]).round(2))
mf = df[df.Gender.isin(["Male", "Female"])].groupby("Gender").Workout_Duration_Mins.mean()
print("Male - Female avg gap:", round(mf["Male"] - mf["Female"], 2))

# 10. Monthly revenue by tier: each member is billed once per month in which they were active
mm = df.drop_duplicates(["Member_ID", "Month"])[["Member_ID", "Month", "Membership_Tier", "Monthly_Fee"]]
rev = mm.pivot_table(index="Month", columns="Membership_Tier", values="Monthly_Fee", aggfunc="sum", fill_value=0)
rev = rev[["Basic", "Gold", "Platinum"]]; rev["Total"] = rev.sum(axis=1)
print("== 10. MONTHLY REVENUE (active-member billing) =="); print(rev.to_string())
print("Average monthly revenue by tier:"); print(rev.mean().round(0))
print("Total revenue over period by tier:"); print(rev.sum())
rev.to_csv(f"{out}/monthly_revenue_by_tier.csv")
# alt (naive) basis for reference: sum of fee on every check-in row
print("Naive sum of fee per check-in (overcounts):", df.Monthly_Fee.sum())
print("Does any member appear in >1 tier?", (df.groupby("Member_ID").Membership_Tier.nunique() > 1).sum())

# 11. Low engagement: < 2 visits per month
n_months = df.Month.nunique()
mem = df.groupby("Member_ID").agg(visits=("Checkin_ID", "count"),
                                  active_months=("Month", "nunique"),
                                  first=("Checkin_Time", "min"), last=("Checkin_Time", "max"),
                                  tier=("Membership_Tier", "first"))
mem["visits_per_active_month"] = mem.visits / mem.active_months
mem["visits_per_calendar_month"] = mem.visits / n_months
mem["low_A"] = mem.visits_per_active_month < 2          # <2 visits in the months they came
mem["low_B"] = mem.visits_per_calendar_month < 2        # <2 visits per month over whole window
mm_counts = df.groupby(["Member_ID", "Month"]).size()
print("== 11. LOW ENGAGEMENT ==")
print("Months in window:", n_months, "| members:", len(mem))
print("A) avg <2 visits per ACTIVE month:", mem.low_A.sum(), f"({mem.low_A.mean():.1%})")
print("B) <2 visits per CALENDAR month over window:", mem.low_B.sum(), f"({mem.low_B.mean():.1%})")
print("Member-months with 1 visit:", (mm_counts == 1).sum(), "of", len(mm_counts), f"({(mm_counts==1).mean():.1%})")
print("Visits per member distribution:"); print(mem.visits.describe())
print("Low-engagement (A) by tier:"); print(mem.groupby("tier").low_A.agg(["sum", "count", "mean"]).round(3))
print("Members with exactly 1 visit ever:", (mem.visits == 1).sum())
mem[mem.low_A].to_csv(f"{out}/low_engagement_members.csv")

# 12. NumPy stats
d = df.Workout_Duration_Mins.to_numpy()
q1, q2, q3 = np.percentile(d, [25, 50, 75])
print("== 12. NUMPY ==")
print("median", np.median(d), "variance(sample)", round(np.var(d, ddof=1), 2),
      "variance(pop)", round(np.var(d), 2), "std", round(np.std(d, ddof=1), 2))
print("Q1,Q2,Q3:", q1, q2, q3)

# 13. IQR outliers
iqr = q3 - q1; lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("== 13. IQR ==", "IQR", iqr, "fences", lo, hi)
df["Outlier"] = np.select([d < lo, d > hi], ["Unusually short", "Unusually long"], "Normal")
print(df.Outlier.value_counts())
print("min/max duration:", d.min(), d.max())

# 14. Intensity labels (documented thresholds)
#   Short < 45 min | Moderate 45-89 min | Extended >= 90 min
df["Intensity"] = np.select([d < 45, d < 90], ["Short", "Moderate"], "Extended")
print("== 14. INTENSITY ==")
print(df.Intensity.value_counts().reindex(["Short", "Moderate", "Extended"]))
print((100 * df.Intensity.value_counts(normalize=True)).round(1).reindex(["Short", "Moderate", "Extended"]))
print(pd.crosstab(df.Membership_Tier, df.Intensity, normalize="index").round(3))
print(pd.crosstab(df.Fitness_Class, df.Intensity, normalize="index").round(3))

# extras for recommendations
print("== EXTRAS ==")
print("Weekend share:", round(df.Weekday.isin(["Saturday", "Sunday"]).mean() * 100, 1))
print("Class vs no class by hour band:"); print(pd.crosstab(band, df.Fitness_Class == "No Class (Open Gym)").to_string())
print("Monthly check-ins:"); print(df.groupby("Month").size().to_string())
print("Class x peak hour top:"); print(df.groupby(["Hour", "Fitness_Class"]).size().sort_values(ascending=False).head(8))

df.to_csv(f"{out}/gym_cleaned.csv", index=False)
tier.to_csv(f"{out}/tier_comparison.csv"); cls.to_csv(f"{out}/class_ranking.csv")


== 1. SHAPE/DTYPES ==
(850, 8)
Checkin_ID                object
Member_ID                 object
Membership_Tier           object
Gender                    object
Checkin_Time              object
Workout_Duration_Mins    float64
Fitness_Class             object
Monthly_Fee                int64
dtype: object
== 2. MISSING ==
Checkin_ID                 0
Member_ID                  0
Membership_Tier            0
Gender                     0
Checkin_Time               0
Workout_Duration_Mins     30
Fitness_Class            133
Monthly_Fee                0
dtype: int64
Duplicate Checkin_ID: 0 | fully duplicated rows: 0
== 3. Median duration by tier (pre-imputation) ==
Membership_Tier
Basic       70.0
Gold        68.0
Platinum    66.5
Name: Workout_Duration_Mins, dtype: float64
Imputed: 30 -> remaining missing: 0
Date range: 2024-03-01 05:00:00 -> 2025-12-30 16:00:00
== 5. TOTALS ==
Total check-ins: 850 | unique members: 299
Avg workout (mins): 68.67
== 6. TIER ==
                 checkins  

# New Section